**Integrantes:** Marvin Francisconi | Ricardo Illanes  
**Curso:** Visualización de Datos (ADY1104)  
**Caso:** StreamView Analytics

## Índice:

* [1. Problema de negocio](#sec-1)
* [2. Audiencia objetivo y propósito comunicacional](#sec-2)
* [3. Descripción e integración de las fuentes de datos](#sec-3)
* [4. EDA](#sec-4)
    * [4.1 Descripción del EDA](#sec-4-1)
* [5. Identificación de patrones, tendencias y hallazgos relevantes](#sec-5)
* [6. Selección y justificación de los gráficos utilizados](#sec-6)
* [7. Construcción del dashboard con KPIs, filtros e interacción](#sec-7)
* [8. Desarrollo de la narrativa visual (Data Storytelling)](#sec-8)
* [9. Conclusiones y recomendaciones basadas en los datos](#sec-9)

### &rarr; Carga de datasets

In [ ]:
import pandas as pd

# Cargar los datasets
usuarios = pd.read_csv("StreamView_Analytics_CSV/usuarios.csv")
contenidos = pd.read_csv("StreamView_Analytics_CSV/contenidos.csv")
suscripciones = pd.read_csv("StreamView_Analytics_CSV/suscripciones.csv")
dispositivos = pd.read_csv("StreamView_Analytics_CSV/dispositivos.csv")
reproducciones = pd.read_csv("StreamView_Analytics_CSV/reproducciones.csv")
calificaciones = pd.read_csv("StreamView_Analytics_CSV/calificaciones.csv")
interacciones = pd.read_csv("StreamView_Analytics_CSV/interacciones.csv")


# Diccionario para trabajar con todos los datasets
datasets = {
    "Usuarios": usuarios,
    "Contenidos": contenidos,
    "Suscripciones": suscripciones,
    "Dispositivos": dispositivos,
    "Reproducciones": reproducciones,
    "Calificaciones": calificaciones,
    "Interacciones": interacciones
}


# Exploración inicial
for nombre, df in datasets.items():
    print("=" * 70)
    print(nombre)
    print("=" * 70)
    
    print(f"Filas: {df.shape[0]}")
    print(f"Columnas: {df.shape[1]}")
    
    print("\nColumnas:")
    print(df.columns.tolist())
    
    print("\nValores nulos:")
    print(df.isnull().sum())
    
    print("\nPrimeras 5 filas:")
    display(df.head())
    
    print()

### Se van a tomar 6 características que se consideran importantes para poder generar un problema de negocio:  
* `Usuario`: País
* `Suscripcines`: Estado
* `Reproducciones`: Dispositivo
* `Calidad`: Calidad de vídeo
* `Contenidos`: Tipo
* `Género`: Género

In [ ]:
print("USUARIOS")
print(usuarios["pais"].value_counts())
print("\nSegmento de edad:")
print(usuarios["segmento_edad"].value_counts())

print("\nSUSCRIPCIONES")
print(suscripciones["plan"].value_counts())
print("\nEstado:")
print(suscripciones["estado"].value_counts())

print("\nREPRODUCCIONES")
print("\nDispositivos:")
print(reproducciones["tipo_dispositivo"].value_counts())

print("\nCalidad:")
print(reproducciones["calidad_video"].value_counts())

print("\nCONTENIDOS")
print("\nTipo:")
print(contenidos["tipo_contenido"].value_counts())

print("\nGénero:")
print(contenidos["genero_principal"].value_counts())

### Resumen corto:  

* 1.500 usuarios, principalmente de Chile, México, Perú, Colombia, Argentina y España.
* La mayor concentración está entre 25 y 44 años.
* 1.085 suscripciones activas y 415 canceladas, aproximadamente 27,7% canceladas.
* El plan más utilizado es Estándar.
* El consumo está concentrado en Móvil, Smart TV y Computador.
* La calidad más utilizada es Full HD, seguida de HD.
* El catálogo tiene principalmente Películas y Series.
* Los géneros más numerosos del catálogo son Drama, Comedia y Acción.

<a id = "sec-1"></a>
# 1. Problema de negocio  

StreamView Analytics necesita comprender los patrones asociados a la permanencia y cancelación de suscripciones, considerando las características de sus usuarios y su comportamiento de consumo, para apoyar decisiones orientadas a la retención y experiencia del cliente.

## Pregunta de análisis:
&rarr; **¿Qué patrones de usuarios, suscripciones y consumo se relacionan con la permanencia o cancelación de las suscripciones?**

<a id="sec-2"></a>
# 2. Audiencia objetivo y propósito comunicacional.

## Audiencia objetivo:  
Equipo de gestión y toma de decisiones de StreamView Analytics.

## Propósito educacional:  
Comunicar de manera clara los principales patrones relacionados con la permanencia y cancelación de suscripciones, utilizando información sobre usuarios, suscripciones y comportamiento de consumo para facilitar la toma de decisiones.

<a id="sec-3"></a>
# 3. Descripción e integración de las fuentes de datos.  

La descripción principal se encuentra en archivo README.txt, pincha [aquí](StreamView_Analytics_CSV/README.txt) para acceder.  

Se utilizaron siete fuentes de datos en formato CSV correspondientes a usuarios, contenidos, suscripciones, dispositivos, reproducciones, calificaciones e interacciones. Las fuentes fueron integradas mediante identificadores comunes como `usuario_id`, `contenido_id` y `dispositivo_id`, permitiendo relacionar las características de los usuarios con sus suscripciones, comportamiento de consumo, contenidos e interacciones.  

&rarr; ***Esta integración permite analizar el comportamiento de los usuarios desde distintas dimensiones y estudiar su relación con la permanencia o cancelación de las suscripciones.***

<a id="sec-4"></a>
# 4. EDA

---
# Exploración de suscripciones
---

In [ ]:
estado_suscripciones = (
    suscripciones["estado"]
    .value_counts()
    .to_frame("cantidad")
)

estado_suscripciones["porcentaje"] = (
    estado_suscripciones["cantidad"]
    / estado_suscripciones["cantidad"].sum()
    * 100
).round(2)

display(estado_suscripciones)

In [ ]:
cancelacion_plan = pd.crosstab(
    suscripciones["plan"],
    suscripciones["estado"]
)

cancelacion_plan["total"] = cancelacion_plan.sum(axis=1)

cancelacion_plan["tasa_cancelacion"] = (
    cancelacion_plan["Cancelada"]
    / cancelacion_plan["total"]
    * 100
).round(2)

display(cancelacion_plan)

In [ ]:
cancelacion_pais = pd.crosstab(
    usuarios["pais"],
    suscripciones["estado"]
)

cancelacion_pais["total"] = cancelacion_pais.sum(axis=1)

cancelacion_pais["tasa_cancelacion"] = (
    cancelacion_pais["Cancelada"]
    / cancelacion_pais["total"]
    * 100
).round(2)

display(
    cancelacion_pais.sort_values(
        "tasa_cancelacion",
        ascending=False
    )
)

In [ ]:
cancelacion_edad = pd.crosstab(
    usuarios["segmento_edad"],
    suscripciones["estado"]
)

cancelacion_edad["total"] = cancelacion_edad.sum(axis=1)

cancelacion_edad["tasa_cancelacion"] = (
    cancelacion_edad["Cancelada"]
    / cancelacion_edad["total"]
    * 100
).round(2)

display(cancelacion_edad)

In [ ]:
cancelacion_canal = pd.crosstab(
    usuarios["canal_adquisicion"],
    suscripciones["estado"]
)

cancelacion_canal["total"] = cancelacion_canal.sum(axis=1)

cancelacion_canal["tasa_cancelacion"] = (
    cancelacion_canal["Cancelada"]
    / cancelacion_canal["total"]
    * 100
).round(2)

display(
    cancelacion_canal.sort_values(
        "tasa_cancelacion",
        ascending=False
    )
)

---
# Consumo por estado de suscripción 
---

In [ ]:
reproducciones_suscripcion = reproducciones.merge(
    suscripciones[["usuario_id", "estado", "plan"]],
    on="usuario_id",
    how="left"
)

display(reproducciones_suscripcion.head())

In [ ]:
consumo_estado = (
    reproducciones_suscripcion
    .groupby("estado")
    .agg(
        reproducciones=("reproduccion_id", "count"),
        minutos_totales=("minutos_reproducidos", "sum"),
        minutos_promedio=("minutos_reproducidos", "mean"),
        completado_promedio=("porcentaje_completado", "mean")
    )
    .round(2)
)

display(consumo_estado)

In [ ]:
reproducciones_por_usuario = (
    reproducciones_suscripcion
    .groupby(["estado", "usuario_id"])
    .size()
    .groupby(level=0)
    .mean()
    .round(2)
)

display(reproducciones_por_usuario)

In [ ]:
minutos_por_usuario = (
    reproducciones_suscripcion
    .groupby(["estado", "usuario_id"])["minutos_reproducidos"]
    .sum()
    .groupby(level=0)
    .mean()
    .round(2)
)

display(minutos_por_usuario)

In [ ]:
completado_por_estado = (
    reproducciones_suscripcion
    .groupby("estado")["porcentaje_completado"]
    .mean()
    .round(2)
)

display(completado_por_estado)

In [ ]:
abandono_estado = pd.crosstab(
    reproducciones_suscripcion["estado"],
    reproducciones_suscripcion["abandono_temprano"],
    normalize="index"
).mul(100).round(2)

display(abandono_estado)

---
# Consumo de contenido según sus tipos
---

In [ ]:
consumo_contenido = reproducciones.merge(
    contenidos,
    on="contenido_id",
    how="left"
)

display(
    consumo_contenido[
        [
            "reproduccion_id",
            "contenido_id",
            "titulo",
            "tipo_contenido",
            "genero_principal",
            "minutos_reproducidos",
            "porcentaje_completado"
        ]
    ].head()
)

In [ ]:
consumo_tipo = (
    consumo_contenido
    .groupby("tipo_contenido")
    .agg(
        reproducciones=("reproduccion_id", "count"),
        minutos_totales=("minutos_reproducidos", "sum"),
        completado_promedio=("porcentaje_completado", "mean")
    )
    .round(2)
    .sort_values("reproducciones", ascending=False)
)

display(consumo_tipo)

In [ ]:
consumo_genero = (
    consumo_contenido
    .groupby("genero_principal")
    .agg(
        reproducciones=("reproduccion_id", "count"),
        minutos_totales=("minutos_reproducidos", "sum"),
        completado_promedio=("porcentaje_completado", "mean")
    )
    .round(2)
    .sort_values("reproducciones", ascending=False)
)

display(consumo_genero)

In [ ]:
top_contenidos = (
    consumo_contenido
    .groupby(["contenido_id", "titulo"])
    .agg(
        reproducciones=("reproduccion_id", "count"),
        minutos_totales=("minutos_reproducidos", "sum"),
        completado_promedio=("porcentaje_completado", "mean")
    )
    .round(2)
    .sort_values("reproducciones", ascending=False)
    .head(10)
)

display(top_contenidos)

In [ ]:
consumo_completo = reproducciones.merge(
    suscripciones[["usuario_id", "estado"]],
    on="usuario_id",
    how="left"
).merge(
    contenidos[
        [
            "contenido_id",
            "tipo_contenido",
            "genero_principal"
        ]
    ],
    on="contenido_id",
    how="left"
)

In [ ]:
genero_estado = pd.crosstab(
    consumo_completo["genero_principal"],
    consumo_completo["estado"],
    normalize="columns"
).mul(100).round(2)

display(
    genero_estado.sort_values(
        "Activa",
        ascending=False
    )
)

---
# Consumo según sus dispositivos
---

In [ ]:
dispositivo_estado = pd.crosstab(
    reproducciones_suscripcion["tipo_dispositivo"],
    reproducciones_suscripcion["estado"],
    normalize="columns"
).mul(100).round(2)

display(dispositivo_estado)

In [ ]:
calidad_estado = pd.crosstab(
    reproducciones_suscripcion["calidad_video"],
    reproducciones_suscripcion["estado"],
    normalize="columns"
).mul(100).round(2)

display(calidad_estado)

In [ ]:
buffering_estado = (
    reproducciones_suscripcion
    .groupby("estado")["buffering_segundos"]
    .agg(
        promedio="mean",
        mediana="median"
    )
    .round(2)
)

display(buffering_estado)

In [ ]:
experiencia_estado = (
    reproducciones_suscripcion
    .groupby("estado")
    .agg(
        reproducciones=("reproduccion_id", "count"),
        minutos_promedio=("minutos_reproducidos", "mean"),
        completado_promedio=("porcentaje_completado", "mean"),
        buffering_promedio=("buffering_segundos", "mean")
    )
    .round(2)
)

display(experiencia_estado)

<a id="sec-4-1"></a>
## 4.1 Descripción del EDA
Se realizó un análisis exploratorio de las principales variables relacionadas con usuarios, suscripciones, consumo de contenidos y experiencia de reproducción. El objetivo fue identificar diferencias y patrones asociados al estado de las suscripciones.

En primer lugar, se observó que el **72,33% de las suscripciones se encuentran activas y el 27,67% canceladas**. Al analizar la cancelación según el plan contratado, el plan Básico presenta la mayor tasa de cancelación, con **44,14%**, seguido por Estándar con **20,67%** y Premium con **11,99%**.

Respecto al comportamiento de consumo, los usuarios con suscripción activa presentan un mayor nivel promedio de actividad: registran aproximadamente **58% más reproducciones por usuario y 60% más minutos de reproducción por usuario** que los usuarios cuya suscripción figura como cancelada. Sin embargo, el porcentaje promedio de contenido completado es prácticamente igual entre ambos grupos (**71,52% frente a 71,58%**).

También se analizaron las preferencias de contenido. Las distribuciones de géneros presentan patrones similares entre usuarios activos y cancelados, por lo que el género de contenido no muestra diferencias relevantes entre ambos grupos. Las películas concentran la mayor cantidad de reproducciones, seguidas por las series y los documentales.

Finalmente, se evaluó la experiencia técnica de reproducción. La distribución de dispositivos es similar entre ambos grupos y el **buffering promedio es prácticamente idéntico** (8,26 segundos en usuarios activos y 8,24 segundos en cancelados). En cuanto a calidad de video, los usuarios activos presentan una mayor proporción de reproducciones en 4K y Full HD, mientras que los cancelados presentan una mayor proporción en HD y SD.

Estos resultados permiten orientar el análisis hacia la identificación de los factores y patrones más asociados a la permanencia o cancelación de las suscripciones, los cuales serán profundizados mediante las visualizaciones y el dashboard.


<a id="sec-5"></a>
# 5. Identificación de patrones, tendencias y hallazgos relevantes

A partir del análisis exploratorio se identificaron cuatro hallazgos principales. 
* En primer lugar, el plan Básico presenta la mayor tasa de cancelación, alcanzando un 44,14%, frente a un 20,67% en Estándar y un 11,99% en Premium.

* En segundo lugar, los usuarios con suscripción activa presentan un mayor nivel de consumo, con aproximadamente 58% más reproducciones y 60% más minutos de reproducción por usuario que los usuarios cancelados.

* En tercer lugar, el porcentaje de contenido completado es prácticamente igual entre ambos grupos, por lo que la diferencia observada se relaciona principalmente con el volumen de consumo y no con una mayor tendencia al abandono de contenidos.

* Finalmente, la experiencia técnica presenta diferencias limitadas. El buffering promedio es prácticamente igual entre usuarios activos y cancelados, mientras que sí se observa una mayor proporción de reproducciones en 4K y Full HD entre los usuarios activos.

Estos hallazgos permiten enfocar la visualización en la relación entre plan, nivel de consumo y estado de la suscripción, dejando la experiencia técnica como un elemento complementario del análisis.

<a id = "sec-6"></a>
# 6. Selección y justificación de los gráficos utilizados.

La selección de los gráficos se realizó a partir de los cuatro hallazgos descritos en la sección anterior. Para cada hallazgo se definió primero **qué debe entender la audiencia** y recién después se eligió la tipología de gráfico, siguiendo el criterio de que el tipo de gráfico se subordina al mensaje y no al revés.

## 6.1 Criterios aplicados

* **Percepción visual y jerarquía:** se utiliza la posición sobre un eje común como codificación principal, por ser el atributo preatentivo más preciso para comparar magnitudes. El título de cada gráfico enuncia la conclusión y no la variable, de modo que la lectura comience por el mensaje.
* **Uso del color:** la paleta es funcional, no decorativa. Se emplea un único color de énfasis (rojo/naranjo) para el elemento que motiva la decisión, gris neutro para el contexto y azul para los valores de referencia. Ningún hallazgo se codifica solo con color.
* **Reducción de carga cognitiva:** se eliminan los bordes superior y derecho, las grillas son tenues, los valores se rotulan directamente sobre las barras y se evitan las leyendas cuando el rótulo puede acompañar al dato.
* **Principios de Gestalt:** proximidad para agrupar las barras de un mismo estado, semejanza de color para indicar pertenencia a las categorías "Activa" o "Cancelada", y continuidad mediante un orden coherente de las categorías en todos los gráficos.
* **Coherencia con la audiencia:** el equipo de gestión necesita decidir sobre retención, por lo que cada gráfico responde a una pregunta accionable y se expresa en tasas de cancelación y consumo por usuario, en lugar de conteos absolutos difíciles de comparar.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ---------------------------------------------------------------
# Paleta y estilo base
# ---------------------------------------------------------------
# Se define una sola vez para que todos los gráficos y el dashboard
# compartan el mismo lenguaje visual (coherencia y menor carga cognitiva).

COLOR_CRITICO = "#D64545"   # el elemento que exige una decisión
COLOR_ALERTA = "#F97316"    # segundo nivel de atención
COLOR_AZUL = "#0067B1"      # valores de referencia / categoría positiva
COLOR_NEUTRO = "#D6E0E8"    # contexto, no compite por la atención
COLOR_TEXTO = "#17324D"
COLOR_SECUNDARIO = "#53677A"
FONDO = "#F3F7FA"

# Orden fijo de los estados en todos los gráficos (principio de continuidad)
ORDEN_ESTADO = ["Activa", "Cancelada"]
COLOR_ESTADO = {"Activa": COLOR_AZUL, "Cancelada": COLOR_ALERTA}


def estilo_ejes(ax, titulo, subtitulo=None, eje_grilla="x"):
    """Aplica el estilo común: título-conclusión a la izquierda,
    grilla tenue y bordes innecesarios ocultos."""

    ax.set_title(
        titulo,
        loc="left",
        fontsize=13,
        fontweight="bold",
        color=COLOR_TEXTO,
        pad=20 if subtitulo else 10
    )

    if subtitulo:
        ax.text(
            0, 1.03, subtitulo,
            transform=ax.transAxes,
            fontsize=9.5,
            color=COLOR_SECUNDARIO
        )

    ax.grid(axis=eje_grilla, alpha=0.16)
    ax.set_axisbelow(True)

    if eje_grilla == "x":
        ax.spines[["top", "right", "left"]].set_visible(False)
    else:
        ax.spines[["top", "right"]].set_visible(False)

    ax.tick_params(colors=COLOR_SECUNDARIO, labelsize=10)

    return ax


# ---------------------------------------------------------------
# Base analítica para las visualizaciones
# ---------------------------------------------------------------
# Se reutilizan los dataframes cargados en la sección 3 y se construye
# una tabla por usuario, que es la unidad de análisis del caso.

base_usuarios = usuarios.merge(suscripciones, on="usuario_id")

reproducciones_base = reproducciones.merge(
    base_usuarios[["usuario_id", "estado", "plan", "pais", "segmento_edad"]],
    on="usuario_id"
)

print("Usuarios con suscripción:", base_usuarios.shape[0])
print("Reproducciones enlazadas:", reproducciones_base.shape[0])

## 6.2 Gráfico 1: la cancelación se concentra en el plan Básico

**Pregunta que responde:** ¿qué plan concentra el riesgo de cancelación?

**Tipología elegida:** barras horizontales ordenadas de mayor a menor, con línea de referencia en la tasa global.

**Justificación:** se comparan tres categorías nominales con una única medida. Las barras horizontales permiten leer las etiquetas de los planes sin rotarlas y ordenar las categorías por magnitud, lo que convierte el ranking en información preatentiva. Se descarta el gráfico circular porque, al tratarse de tres proporciones que no suman un total común, obligaría a comparar ángulos, una tarea perceptualmente menos precisa que comparar longitudes sobre una línea base compartida. El color se usa como énfasis y no como codificación: solo el plan Básico se destaca en rojo porque es el que exige una decisión, mientras los otros dos quedan en gris para aportar contexto sin competir por la atención.

In [ ]:
cancelacion_plan = (
    pd.crosstab(base_usuarios["plan"], base_usuarios["estado"], normalize="index")
    .mul(100)
    .sort_values("Cancelada")
)

tasa_global = (base_usuarios["estado"] == "Cancelada").mean() * 100

# Un solo plan se destaca: el que concentra el problema.
colores = [
    COLOR_CRITICO if valor == cancelacion_plan["Cancelada"].max() else COLOR_NEUTRO
    for valor in cancelacion_plan["Cancelada"]
]

fig, ax = plt.subplots(figsize=(9, 4), facecolor=FONDO)
ax.set_facecolor(FONDO)

ax.barh(cancelacion_plan.index, cancelacion_plan["Cancelada"],
        color=colores, height=0.6)

ax.axvline(tasa_global, color=COLOR_AZUL, linestyle="--", linewidth=1.4)
ax.text(tasa_global + 0.6, 2.34, f"Tasa global {tasa_global:.1f}%",
        color=COLOR_AZUL, fontsize=9, va="center")

for plan, valor in cancelacion_plan["Cancelada"].items():
    ax.text(valor + 0.8, plan, f"{valor:.1f}%", va="center",
            fontsize=11, fontweight="bold", color=COLOR_TEXTO)

ax.set_xlim(0, 52)
ax.set_xlabel("Suscripciones canceladas (%)", color=COLOR_SECUNDARIO)

estilo_ejes(
    ax,
    "El plan Básico cancela más del doble que el promedio de la plataforma",
    f"{cancelacion_plan['Cancelada'].max():.1f}% de cancelación frente a "
    f"{cancelacion_plan['Cancelada'].min():.1f}% en Premium"
)

plt.tight_layout()
plt.show()

## 6.3 Gráfico 2: quien cancela consumía menos antes de cancelar

**Pregunta que responde:** ¿cuánta diferencia de consumo existe entre quien permanece y quien cancela?

**Tipología elegida:** dos paneles de barras verticales pareadas, una métrica por panel.

**Justificación:** se comparan dos grupos en dos métricas con unidades distintas (cantidad de reproducciones y minutos). Separarlas en dos paneles evita representar escalas incompatibles sobre un mismo eje, lo que distorsionaría la comparación. Dentro de cada panel, la proximidad agrupa el par y la semejanza de color mantiene la identidad de cada estado, de modo que la audiencia no necesita volver a la leyenda en el segundo panel. La diferencia porcentual se anota directamente sobre el gráfico porque es el dato que se debe recordar, no los valores absolutos.

In [ ]:
usuarios_por_estado = base_usuarios.groupby("estado")["usuario_id"].nunique()

consumo_estado_usuario = pd.DataFrame({
    "Reproducciones por usuario":
        reproducciones_base.groupby("estado").size() / usuarios_por_estado,
    "Minutos por usuario":
        reproducciones_base.groupby("estado")["minutos_reproducidos"].sum()
        / usuarios_por_estado
}).reindex(ORDEN_ESTADO)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), facecolor=FONDO)

for ax, metrica in zip(axes, consumo_estado_usuario.columns):
    valores = consumo_estado_usuario[metrica]
    ax.set_facecolor(FONDO)

    ax.bar(valores.index, valores,
           color=[COLOR_ESTADO[estado] for estado in valores.index],
           width=0.5)

    for estado, valor in valores.items():
        texto = f"{valor:.1f}" if valor < 100 else f"{valor:,.0f}".replace(",", ".")
        ax.text(estado, valor * 1.02, texto,
                ha="center", fontsize=12, fontweight="bold", color=COLOR_TEXTO)

    brecha = (valores["Activa"] / valores["Cancelada"] - 1) * 100

    ax.set_ylim(0, valores.max() * 1.25)
    ax.set_ylabel(metrica, color=COLOR_SECUNDARIO)

    estilo_ejes(ax, metrica, f"Los activos consumen {brecha:.0f}% más",
                eje_grilla="y")

fig.suptitle(
    "El consumo previo, y no el tipo de contenido, separa a quien permanece de quien cancela",
    x=0.015, ha="left", fontsize=14, fontweight="bold", color=COLOR_TEXTO
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 6.4 Gráfico 3: el porcentaje completado no distingue a los dos grupos

**Pregunta que responde:** ¿la diferencia se explica por abandono de contenidos o por volumen de consumo?

**Tipología elegida:** barras horizontales con el eje completo de 0 a 100% y la diferencia anotada.

**Justificación:** este gráfico se incluye deliberadamente para **descartar** una hipótesis, y por eso su diseño se diferencia del resto. El eje parte en cero y llega a 100 porque truncarlo exageraría visualmente una diferencia de 0,07 puntos porcentuales y llevaría a la audiencia a una conclusión falsa. Ambas barras se mantienen en gris neutro: si no hay hallazgo, no corresponde usar color de énfasis. Es la forma honesta de mostrar una ausencia de diferencia y permite atribuir la brecha de la sección anterior al volumen de consumo, y no a una menor tolerancia al contenido.

In [ ]:
completado_estado = (
    reproducciones_base.groupby("estado")["porcentaje_completado"]
    .mean()
    .reindex(ORDEN_ESTADO)
)

diferencia = abs(completado_estado["Activa"] - completado_estado["Cancelada"])

fig, ax = plt.subplots(figsize=(9, 3.4), facecolor=FONDO)
ax.set_facecolor(FONDO)

# Sin hallazgo, sin color de énfasis: ambas barras en gris neutro.
ax.barh(completado_estado.index, completado_estado,
        color=COLOR_NEUTRO, height=0.5)

for estado, valor in completado_estado.items():
    ax.text(valor + 1.2, estado, f"{valor:.2f}%", va="center",
            fontsize=11, fontweight="bold", color=COLOR_TEXTO)

ax.set_xlim(0, 100)
ax.invert_yaxis()  # mantiene el orden Activa / Cancelada de arriba hacia abajo
ax.set_xlabel("Contenido completado en promedio (%)", color=COLOR_SECUNDARIO)

estilo_ejes(
    ax,
    "Cuando ven contenido, ambos grupos lo completan igual",
    f"Diferencia de solo {diferencia:.2f} puntos porcentuales: ".replace(".", ",", 1) +
    "el problema es cuánto ven, no cómo lo ven"
)

plt.tight_layout()
plt.show()

## 6.5 Gráfico 4: la experiencia técnica aporta una señal secundaria

**Pregunta que responde:** ¿la calidad de reproducción acompaña al riesgo de cancelación?

**Tipología elegida:** barras apiladas al 100% con una escala secuencial de calidad.

**Justificación:** la variable de calidad es **ordinal** (SD, HD, Full HD, 4K) y lo que interesa es la composición interna de cada grupo, no el volumen absoluto. Las barras apiladas al 100% son adecuadas porque el total de cada estado carece de significado comparativo y lo relevante es la proporción. Se usa una escala secuencial de un solo tono, de claro a oscuro, que respeta el orden natural de la variable: una paleta categórica de colores distintos obligaría a memorizar una leyenda arbitraria. El hallazgo se mantiene como complementario y así se declara en el subtítulo, para no darle más peso visual del que la evidencia sostiene.

In [ ]:
orden_calidad = ["SD", "HD", "Full HD", "4K"]

calidad_estado = (
    pd.crosstab(reproducciones_base["estado"],
                reproducciones_base["calidad_video"],
                normalize="index")
    .mul(100)
    .reindex(index=ORDEN_ESTADO, columns=orden_calidad)
)

# Escala secuencial: la variable es ordinal, el color también debe serlo.
escala_calidad = ["#CFE0EC", "#8FB8D6", "#3D85BC", "#0B4C82"]

fig, ax = plt.subplots(figsize=(10, 3.6), facecolor=FONDO)
ax.set_facecolor(FONDO)

acumulado = calidad_estado[orden_calidad[0]] * 0

for calidad, color in zip(orden_calidad, escala_calidad):
    valores = calidad_estado[calidad]

    ax.barh(calidad_estado.index, valores, left=acumulado,
            color=color, height=0.55, label=calidad)

    for estado, valor in valores.items():
        ax.text(acumulado[estado] + valor / 2, estado, f"{valor:.0f}%",
                ha="center", va="center", fontsize=9.5, fontweight="bold",
                color="white" if calidad in ("Full HD", "4K") else COLOR_TEXTO)

    acumulado = acumulado + valores

alta_activa = calidad_estado.loc["Activa", ["Full HD", "4K"]].sum()
alta_cancelada = calidad_estado.loc["Cancelada", ["Full HD", "4K"]].sum()

ax.set_xlim(0, 100)
ax.invert_yaxis()  # Activa arriba, igual que en el resto de los gráficos
ax.set_xlabel("Distribución de reproducciones (%)", color=COLOR_SECUNDARIO)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=4,
          frameon=False, fontsize=9.5)

estilo_ejes(
    ax,
    "Los usuarios que cancelan reproducen más seguido en calidad baja",
    f"Full HD y 4K concentran {alta_activa:.0f}% en activos frente a "
    f"{alta_cancelada:.0f}% en cancelados (señal complementaria, no determinante)"
)

plt.tight_layout()
plt.show()

## 6.6 Gráfico 5: el motivo declarado confirma el diagnóstico

**Pregunta que responde:** ¿qué razón declaran los usuarios al cancelar y es consistente con los datos de consumo?

**Tipología elegida:** barras horizontales ordenadas, con dos categorías destacadas.

**Justificación:** son seis categorías nominales sin orden natural, con etiquetas de distinta longitud y una distribución desigual: las barras horizontales ordenadas por frecuencia resuelven los tres problemas a la vez. Se destacan en color solo los dos motivos que el resto del análisis permite accionar, "Poco uso" y "Precio", porque son los que se conectan con la brecha de consumo y con la concentración del problema en el plan más económico. Este gráfico cierra el argumento: lo que muestra el comportamiento y lo que declara el usuario apuntan en la misma dirección.

In [ ]:
motivos = base_usuarios["motivo_cancelacion"].value_counts().sort_values()

total_cancelados = int(motivos.sum())
destacados = ["Poco uso", "Precio"]

colores_motivo = [
    COLOR_CRITICO if motivo == "Poco uso"
    else COLOR_ALERTA if motivo in destacados
    else COLOR_NEUTRO
    for motivo in motivos.index
]

fig, ax = plt.subplots(figsize=(9.5, 4.2), facecolor=FONDO)
ax.set_facecolor(FONDO)

ax.barh(motivos.index, motivos, color=colores_motivo, height=0.65)

for motivo, valor in motivos.items():
    ax.text(valor + 2, motivo,
            f"{valor}  ({valor / total_cancelados * 100:.0f}%)",
            va="center", fontsize=10, color=COLOR_TEXTO)

ax.set_xlim(0, motivos.max() * 1.32)
ax.set_xlabel("Suscripciones canceladas", color=COLOR_SECUNDARIO)

peso_accionable = motivos[destacados].sum() / total_cancelados * 100

estilo_ejes(
    ax,
    "El motivo más declarado es el poco uso, no el contenido ni la tecnología",
    f"Poco uso y precio explican el {peso_accionable:.0f}% de las "
    f"{total_cancelados} cancelaciones"
)

plt.tight_layout()
plt.show()

## 6.7 Gráfico 6: la renovación automática como variable de control

**Pregunta que responde:** ¿existe alguna variable operativa asociada de forma directa a la cancelación?

**Tipología elegida:** barras verticales de dos categorías con contraste máximo, acompañadas de un panel de concentración por plan.

**Justificación:** con solo dos categorías y una diferencia extrema, cualquier codificación más elaborada agregaría carga cognitiva sin agregar información. Se usa el contraste directo entre el color crítico y el azul de referencia, y los valores se anotan sobre las barras para que el gráfico se lea sin consultar el eje. El segundo panel repite la tipología del Gráfico 1 de forma intencional: al mantener la misma forma y la misma lógica de color, la audiencia reconoce de inmediato que el problema vuelve a concentrarse en el plan Básico. Se presenta como **variable de control** y no como hallazgo causal, ya que la ausencia de renovación automática está asociada al término de la suscripción casi por definición; su valor está en identificar un segmento operativo sobre el cual intervenir de forma anticipada.

In [ ]:
renovacion = (
    pd.crosstab(base_usuarios["renovacion_automatica"],
                base_usuarios["estado"], normalize="index")
    .mul(100)["Cancelada"]
    .sort_values()
)

etiquetas = [f"Renovación automática: {valor}" for valor in renovacion.index]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), facecolor=FONDO,
                         gridspec_kw={"width_ratios": [1.05, 1]})

# Panel 1: tasa de cancelación según renovación automática
ax = axes[0]
ax.set_facecolor(FONDO)
ax.bar(etiquetas, renovacion, color=[COLOR_AZUL, COLOR_CRITICO], width=0.5)

for etiqueta, valor in zip(etiquetas, renovacion):
    ax.text(etiqueta, valor + 2.5, f"{valor:.1f}%", ha="center",
            fontsize=12, fontweight="bold", color=COLOR_TEXTO)

ax.set_ylim(0, 118)
ax.set_ylabel("Suscripciones canceladas (%)", color=COLOR_SECUNDARIO)
estilo_ejes(ax, "Sin renovación automática, la suscripción termina",
            eje_grilla="y")

# Panel 2: dónde se concentra la falta de renovación automática
sin_renovacion = (
    pd.crosstab(base_usuarios["plan"],
                base_usuarios["renovacion_automatica"], normalize="index")
    .mul(100)["No"]
    .sort_values()
)

ax = axes[1]
ax.set_facecolor(FONDO)

colores_plan = [
    COLOR_CRITICO if valor == sin_renovacion.max() else COLOR_NEUTRO
    for valor in sin_renovacion
]

ax.barh(sin_renovacion.index, sin_renovacion, color=colores_plan, height=0.55)

for plan, valor in sin_renovacion.items():
    ax.text(valor + 1, plan, f"{valor:.1f}%", va="center",
            fontsize=11, fontweight="bold", color=COLOR_TEXTO)

ax.set_xlim(0, sin_renovacion.max() * 1.35)
ax.set_xlabel("Usuarios sin renovación automática (%)", color=COLOR_SECUNDARIO)
estilo_ejes(ax, "Y se concentra otra vez en el plan Básico")

plt.tight_layout()
plt.show()

## 6.8 Síntesis de la selección y gráficos descartados

| # | Hallazgo que comunica | Tipología seleccionada | Codificación principal | Por qué esa y no otra |
|---|---|---|---|---|
| G1 | Cancelación por plan | Barras horizontales ordenadas | Longitud sobre línea base común | Un circular obligaría a comparar ángulos entre proporciones que no comparten total |
| G2 | Brecha de consumo | Barras pareadas en dos paneles | Longitud + agrupación por proximidad | Dos métricas con unidades distintas no pueden compartir un mismo eje |
| G3 | Ausencia de diferencia en completado | Barras horizontales con eje 0–100 | Longitud sin truncar | Truncar el eje exageraría una diferencia de 0,07 pp e induciría a error |
| G4 | Composición de calidad de video | Barras apiladas al 100% | Proporción + escala secuencial | La variable es ordinal; una paleta categórica rompería ese orden |
| G5 | Motivo declarado de cancelación | Barras horizontales ordenadas | Longitud + énfasis selectivo | Seis categorías nominales con etiquetas largas no caben en barras verticales |
| G6 | Renovación automática | Barras verticales de dos categorías | Contraste de color | Con dos valores extremos, cualquier codificación adicional sería ruido |

**Gráficos evaluados y descartados:**

* **Gráfico circular del estado de suscripción:** aporta una sola proporción que el KPI del dashboard comunica con menos tinta.
* **Distribución de género de contenido por estado:** las diferencias son inferiores a un punto porcentual en todas las categorías; graficarlas sugeriría un patrón inexistente.
* **Distribución de dispositivos y buffering promedio:** el comportamiento es equivalente entre ambos grupos (8,26 frente a 8,24 segundos), por lo que se documentan en el EDA pero no se visualizan.
* **Mapa por país:** las diferencias entre países son menores que la variación por plan, y un mapa dedicaría una superficie visual amplia a la señal más débil del análisis.

<a id="sec-7"></a>
# 7. Construcción del dashboard con KPIs, filtros e interacción.

El dashboard reúne en una sola vista los gráficos justificados en la sección anterior, organizados según una **jerarquía visual de tres niveles de lectura**:

1. **Nivel de titular:** el título del dashboard enuncia la conclusión y los cuatro KPIs entregan la magnitud del problema en menos de cinco segundos de lectura.
2. **Nivel de diagnóstico:** la banda central responde *dónde* está el problema (plan) y *por qué* ocurre (brecha de consumo). Ocupa la mayor superficie porque concentra la decisión.
3. **Nivel de detalle y acción:** la banda inferior entrega el motivo declarado y la tabla de segmentos priorizados, que es la salida operativa del análisis.

La lectura sigue el recorrido natural en Z de una audiencia occidental: KPIs de izquierda a derecha, diagnóstico y luego detalle. El espacio en blanco entre bandas actúa como separador por proximidad, de modo que cada nivel se perciba como un bloque sin necesidad de recuadros ni líneas divisorias adicionales.

## 7.1 Interacción y filtros

El dashboard se implementa como una **función parametrizada**: `construir_dashboard(pais, segmento_edad, plan)` recalcula todos los KPIs y todos los gráficos sobre el subconjunto filtrado y vuelve a dibujar la vista completa. El panel de filtros, ubicado en la banda inferior derecha, informa en todo momento qué recorte está activo y sobre cuántos usuarios se calculan las cifras, evitando que la audiencia interprete una vista filtrada como la vista global.

> Sobre esta misma función se puede montar interacción con controles gráficos agregando `ipywidgets` (`interact(construir_dashboard, pais=..., segmento_edad=..., plan=...)`). Se mantiene la versión en `matplotlib` porque garantiza que el dashboard se visualice igual en cualquier entorno de corrección, sin dependencias adicionales.

In [ ]:
def construir_dashboard(pais="Todos", segmento_edad="Todos", plan="Todos"):
    """Dibuja el dashboard completo sobre el subconjunto filtrado.

    Los tres parámetros actúan como filtros: al cambiarlos se recalculan
    los KPIs y los cuatro gráficos, manteniendo el mismo diseño."""

    # -----------------------------------------------------------
    # 1. Aplicación de filtros
    # -----------------------------------------------------------
    usuarios_f = base_usuarios.copy()

    if pais != "Todos":
        usuarios_f = usuarios_f[usuarios_f["pais"] == pais]
    if segmento_edad != "Todos":
        usuarios_f = usuarios_f[usuarios_f["segmento_edad"] == segmento_edad]
    if plan != "Todos":
        usuarios_f = usuarios_f[usuarios_f["plan"] == plan]

    if usuarios_f.empty:
        print("El filtro seleccionado no devuelve usuarios.")
        return

    repro_f = reproducciones_base[
        reproducciones_base["usuario_id"].isin(usuarios_f["usuario_id"])
    ]

    # -----------------------------------------------------------
    # 2. KPIs
    # -----------------------------------------------------------
    total_usuarios = usuarios_f["usuario_id"].nunique()
    cancelados = (usuarios_f["estado"] == "Cancelada").sum()
    tasa_cancelacion = cancelados / total_usuarios * 100

    usuarios_estado = usuarios_f.groupby("estado")["usuario_id"].nunique()
    repro_por_usuario = (repro_f.groupby("estado").size() / usuarios_estado)

    ingreso_perdido = usuarios_f.loc[
        usuarios_f["estado"] == "Cancelada", "precio_mensual_usd"
    ].sum()

    # -----------------------------------------------------------
    # 3. Lienzo y jerarquía espacial
    # -----------------------------------------------------------
    fig = plt.figure(figsize=(15, 10), facecolor=FONDO)
    gs = GridSpec(3, 4, figure=fig,
                  height_ratios=[0.7, 2.4, 1.6],
                  hspace=0.55, wspace=0.42)

    kpis = [
        (f"{total_usuarios:,}".replace(",", "."), "Suscripciones analizadas",
         COLOR_TEXTO),
        (f"{tasa_cancelacion:.1f}%", "Tasa de cancelación",
         COLOR_CRITICO if tasa_cancelacion > 25 else COLOR_AZUL),
        (f"{repro_por_usuario.get('Cancelada', 0):.1f}",
         "Reproducciones por usuario que cancela", COLOR_ALERTA),
        (f"US$ {ingreso_perdido:,.0f}".replace(",", "."),
         "Ingreso mensual perdido", COLOR_CRITICO),
    ]

    for columna, (valor, etiqueta, color) in enumerate(kpis):
        ax_kpi = fig.add_subplot(gs[0, columna])
        ax_kpi.axis("off")
        ax_kpi.text(0, 0.60, valor, fontsize=26, fontweight="bold", color=color)
        ax_kpi.text(0, 0.10, etiqueta, fontsize=10.5, color=COLOR_SECUNDARIO)

    # -----------------------------------------------------------
    # 4. Diagnóstico: dónde y por qué
    # -----------------------------------------------------------
    ax_plan = fig.add_subplot(gs[1, :2])
    ax_plan.set_facecolor(FONDO)

    cancel_plan = (
        pd.crosstab(usuarios_f["plan"], usuarios_f["estado"], normalize="index")
        .mul(100)
        .reindex(columns=["Activa", "Cancelada"])
        .fillna(0)
        .sort_values("Cancelada")
    )

    colores_plan = [
        COLOR_CRITICO if valor == cancel_plan["Cancelada"].max() else COLOR_NEUTRO
        for valor in cancel_plan["Cancelada"]
    ]

    ax_plan.barh(cancel_plan.index, cancel_plan["Cancelada"],
                 color=colores_plan, height=0.55)
    ax_plan.axvline(tasa_cancelacion, color=COLOR_AZUL,
                    linestyle="--", linewidth=1.3)

    for nombre_plan, valor in cancel_plan["Cancelada"].items():
        ax_plan.text(valor + 0.8, nombre_plan, f"{valor:.1f}%", va="center",
                     fontsize=10.5, fontweight="bold", color=COLOR_TEXTO)

    ax_plan.set_xlim(0, max(cancel_plan["Cancelada"].max() * 1.3, 10))
    # Limite explicito: con un solo plan filtrado la barra ocuparia todo el panel
    ax_plan.set_ylim(-0.6, len(cancel_plan) - 0.4)
    ax_plan.set_xlabel("Suscripciones canceladas (%)", color=COLOR_SECUNDARIO)
    estilo_ejes(ax_plan, "Dónde se concentra la cancelación",
                f"Línea de referencia: {tasa_cancelacion:.1f}% del recorte actual")

    ax_consumo = fig.add_subplot(gs[1, 2:])
    ax_consumo.set_facecolor(FONDO)

    consumo = repro_por_usuario.reindex(ORDEN_ESTADO).fillna(0)
    ax_consumo.bar(consumo.index, consumo,
                   color=[COLOR_ESTADO[estado] for estado in consumo.index],
                   width=0.45)

    for estado, valor in consumo.items():
        ax_consumo.text(estado, valor * 1.03, f"{valor:.1f}", ha="center",
                        fontsize=12, fontweight="bold", color=COLOR_TEXTO)

    if consumo.get("Cancelada", 0) > 0:
        brecha = (consumo["Activa"] / consumo["Cancelada"] - 1) * 100
        subtitulo_consumo = f"Los activos consumen {brecha:.0f}% más"
    else:
        subtitulo_consumo = "Sin datos de consumo en el recorte"

    ax_consumo.set_ylim(0, max(consumo.max() * 1.25, 1))
    ax_consumo.set_ylabel("Reproducciones por usuario", color=COLOR_SECUNDARIO)
    estilo_ejes(ax_consumo, "Por qué ocurre: brecha de consumo",
                subtitulo_consumo, eje_grilla="y")

    # -----------------------------------------------------------
    # 5. Detalle y acción
    # -----------------------------------------------------------
    ax_motivos = fig.add_subplot(gs[2, :2])
    ax_motivos.set_facecolor(FONDO)

    motivos_f = usuarios_f["motivo_cancelacion"].value_counts().sort_values()

    if not motivos_f.empty:
        colores_mot = [
            COLOR_CRITICO if motivo == motivos_f.idxmax() else COLOR_NEUTRO
            for motivo in motivos_f.index
        ]
        ax_motivos.barh(motivos_f.index, motivos_f, color=colores_mot, height=0.6)

        for motivo, valor in motivos_f.items():
            ax_motivos.text(valor + motivos_f.max() * 0.02, motivo, str(valor),
                            va="center", fontsize=9.5, color=COLOR_TEXTO)

        ax_motivos.set_xlim(0, motivos_f.max() * 1.25)
        ax_motivos.set_ylim(-0.6, len(motivos_f) - 0.4)

    ax_motivos.set_xlabel("Suscripciones canceladas", color=COLOR_SECUNDARIO)
    estilo_ejes(ax_motivos, "Motivo declarado al cancelar")

    # Panel de filtros activos + segmentos priorizados
    ax_panel = fig.add_subplot(gs[2, 2:])
    ax_panel.axis("off")

    ax_panel.text(0, 1.02, "Filtros activos", fontsize=12,
                  fontweight="bold", color=COLOR_TEXTO)
    ax_panel.text(0, 0.86,
                  f"País: {pais}    |    Segmento: {segmento_edad}    |    Plan: {plan}",
                  fontsize=10, color=COLOR_SECUNDARIO)

    prioridad = (
        pd.crosstab(usuarios_f["segmento_edad"], usuarios_f["estado"],
                    normalize="index")
        .mul(100)
        .reindex(columns=["Activa", "Cancelada"])
        .fillna(0)
        .sort_values("Cancelada", ascending=False)
        .head(3)
        .reset_index()
    )

    prioridad["Cancelada"] = prioridad["Cancelada"].map(lambda x: f"{x:.1f}%")
    prioridad = prioridad[["segmento_edad", "Cancelada"]]
    prioridad.columns = ["Segmento de edad", "Cancelación"]

    tabla = ax_panel.table(cellText=prioridad.values,
                           colLabels=prioridad.columns,
                           cellLoc="center", colLoc="center",
                           bbox=[0, 0.05, 1, 0.65])
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)

    ax_panel.text(0, 0.76, "Segmentos priorizados dentro del recorte",
                  fontsize=10.5, fontweight="bold", color=COLOR_TEXTO)

    # -----------------------------------------------------------
    # 6. Titular del dashboard
    # -----------------------------------------------------------
    plan_critico = cancel_plan["Cancelada"].idxmax()

    fig.suptitle(
        f"Retención StreamView: el plan {plan_critico} concentra el riesgo de cancelación",
        x=0.055, y=0.985, ha="left",
        fontsize=19, fontweight="bold", color=COLOR_TEXTO
    )
    fig.text(0.055, 0.945,
             f"{cancelados} cancelaciones sobre {total_usuarios} suscripciones  ·  "
             f"US$ {ingreso_perdido:,.0f}".replace(",", ".") +
             " de ingreso mensual perdido",
             fontsize=11, color=COLOR_SECUNDARIO)

    plt.show()


# Vista general, sin filtros aplicados
construir_dashboard()

### 7.2 Demostración de la interacción

Se aplica el filtro sobre el segmento de edad **18-24**, que en el análisis exploratorio presentaba la mayor tasa de cancelación. Todos los KPIs, los tres gráficos y la tabla de priorización se recalculan sobre ese subconjunto, y el panel inferior derecho deja constancia del recorte aplicado.

In [ ]:
# Misma vista, recalculada sobre el segmento de mayor riesgo
construir_dashboard(segmento_edad="18-24")

In [ ]:
# Filtro combinado: el plan crítico dentro del segmento crítico
construir_dashboard(segmento_edad="18-24", plan="Básico")

<a id="sec-8"></a>
# 8. Desarrollo de la narrativa visual (Data Storytelling)

La narrativa se estructura en tres actos, siguiendo la secuencia **contexto → conflicto → resolución**. Cada acto corresponde a un panel de la figura siguiente, ordenados de izquierda a derecha para que la lectura avance en el mismo sentido que el argumento.

**Acto 1 — Contexto: la situación de partida.**
De cada diez suscripciones de StreamView, casi tres terminan canceladas. Es una cifra que la plataforma conoce, pero que por sí sola no indica dónde intervenir: describe el tamaño del problema, no su ubicación.

**Acto 2 — Conflicto: el problema no está repartido.**
Al abrir esa cifra por plan aparece el quiebre. El plan Básico cancela a más del doble de la tasa global, mientras Premium se mantiene muy por debajo. Y cuando se observa el comportamiento previo a la cancelación, el patrón se vuelve nítido: quienes cancelan consumían cerca de un 58% menos de reproducciones y un 60% menos de minutos. La tensión del relato está en que **no se trata de un problema de contenido ni de tecnología**: quienes cancelan completan el contenido que ven exactamente en la misma proporción que quienes se quedan, y su experiencia de buffering es prácticamente idéntica. El motivo que ellos mismos declaran lo confirma: la razón más frecuente es "poco uso".

**Acto 3 — Resolución: la cancelación se anticipa, no se lamenta.**
Si el problema es el uso y no el producto, entonces es observable antes de que ocurra la cancelación. Un usuario del plan Básico con consumo bajo y sin renovación automática es un caso identificable con semanas de anticipación. Llevar la tasa de cancelación del plan Básico al nivel del plan Estándar es el escenario que se cuantifica en el tercer panel.

In [ ]:
# ---------------------------------------------------------------
# Narrativa visual en tres actos
# ---------------------------------------------------------------

fig = plt.figure(figsize=(16, 6.6), facecolor=FONDO)
gs = GridSpec(1, 3, figure=fig, wspace=0.26,
              left=0.05, right=0.97, top=0.72, bottom=0.20)

# ---- ACTO 1: contexto ----------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(FONDO)

estado_global = (
    base_usuarios["estado"].value_counts(normalize=True).mul(100)
    .reindex(ORDEN_ESTADO)
)

izquierda = 0
for estado in ORDEN_ESTADO:
    color = COLOR_NEUTRO if estado == "Activa" else COLOR_CRITICO
    ax1.barh(["Suscripciones"], estado_global[estado], left=izquierda,
             color=color, height=0.35)
    ax1.text(izquierda + estado_global[estado] / 2, 0,
             f"{estado}\n{estado_global[estado]:.1f}%",
             ha="center", va="center", fontsize=11, fontweight="bold",
             color=COLOR_TEXTO if estado == "Activa" else "white")
    izquierda += estado_global[estado]

ax1.set_xlim(0, 100)
ax1.set_ylim(-0.75, 0.75)  # aire alrededor de la barra: evita que llene el panel
ax1.set_yticks([])
ax1.set_xticks([])
ax1.spines[:].set_visible(False)
ax1.set_title("1. Casi 3 de cada 10 cancelan",
              loc="left", fontsize=13, fontweight="bold", color=COLOR_TEXTO,
              pad=20)
ax1.text(0, 1.04, "Contexto: el tamaño del problema",
         transform=ax1.transAxes, fontsize=9.5, color=COLOR_SECUNDARIO)
ax1.text(0, -0.22,
         "Pero una tasa global no indica\ndónde intervenir.",
         transform=ax1.transAxes, fontsize=10.5, color=COLOR_SECUNDARIO,
         style="italic", va="top")

# ---- ACTO 2: conflicto ---------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(FONDO)

colores_acto2 = [
    COLOR_CRITICO if valor == cancelacion_plan["Cancelada"].max() else COLOR_NEUTRO
    for valor in cancelacion_plan["Cancelada"]
]

ax2.barh(cancelacion_plan.index, cancelacion_plan["Cancelada"],
         color=colores_acto2, height=0.55)
ax2.axvline(tasa_global, color=COLOR_AZUL, linestyle="--", linewidth=1.3)

for nombre_plan, valor in cancelacion_plan["Cancelada"].items():
    ax2.text(valor + 1, nombre_plan, f"{valor:.1f}%", va="center",
             fontsize=10.5, fontweight="bold", color=COLOR_TEXTO)

ax2.set_xlim(0, 55)
estilo_ejes(ax2, "2. Se concentra en el plan Básico",
            "Conflicto: quienes cancelan ya consumían menos")
ax2.text(0, -0.22,
         "58% menos reproducciones, pero mismo\nporcentaje completado y mismo buffering.",
         transform=ax2.transAxes, fontsize=10.5, color=COLOR_SECUNDARIO,
         style="italic", va="top")

# ---- ACTO 3: resolución --------------------------------------
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(FONDO)

usuarios_basico = (base_usuarios["plan"] == "Básico").sum()
tasa_basico = cancelacion_plan.loc["Básico", "Cancelada"]
tasa_estandar = cancelacion_plan.loc["Estándar", "Cancelada"]

cancelaciones_actuales = usuarios_basico * tasa_basico / 100
cancelaciones_objetivo = usuarios_basico * tasa_estandar / 100
recuperables = cancelaciones_actuales - cancelaciones_objetivo

precio_basico = base_usuarios.loc[
    base_usuarios["plan"] == "Básico", "precio_mensual_usd"
].mean()
ingreso_recuperable = recuperables * precio_basico * 12

escenarios = ["Situación actual", "Básico al nivel\nde Estándar"]
valores_escenario = [cancelaciones_actuales, cancelaciones_objetivo]

ax3.bar(escenarios, valores_escenario,
        color=[COLOR_CRITICO, COLOR_AZUL], width=0.45)

for escenario, valor in zip(escenarios, valores_escenario):
    ax3.text(escenario, valor + 5, f"{valor:.0f}", ha="center",
             fontsize=13, fontweight="bold", color=COLOR_TEXTO)

ax3.annotate(
    f"{recuperables:.0f} suscripciones\nrecuperables",
    xy=(0.80, cancelaciones_objetivo * 1.22),
    xytext=(0.5, cancelaciones_actuales * 0.62), va="bottom",
    fontsize=11, fontweight="bold", color=COLOR_AZUL, ha="center",
    arrowprops=dict(arrowstyle="->", color=COLOR_AZUL, linewidth=1.5)
)

ax3.set_ylim(0, cancelaciones_actuales * 1.35)
ax3.set_ylabel("Cancelaciones en el plan Básico", color=COLOR_SECUNDARIO)
estilo_ejes(ax3, "3. El bajo uso se anticipa",
            "Resolución: igualar la tasa del plan Estándar", eje_grilla="y")
ax3.text(0, -0.22,
         "Equivale a US$ " + f"{ingreso_recuperable:,.0f}".replace(",", ".") +
         "\nanuales de ingreso retenido.",
         transform=ax3.transAxes, fontsize=10.5, color=COLOR_SECUNDARIO,
         style="italic", va="top")

fig.suptitle(
    "De una tasa de cancelación general a una acción concreta sobre el plan Básico",
    x=0.05, y=0.94, ha="left", fontsize=17, fontweight="bold", color=COLOR_TEXTO
)
fig.text(0.05, 0.875,
         "Contexto  →  Conflicto  →  Resolución",
         fontsize=11, color=COLOR_SECUNDARIO, fontweight="bold")

plt.show()

### 8.1 Decisiones de diseño de la narrativa

* **Repetición intencional de formas:** el segundo acto reutiliza exactamente la misma tipología y la misma lógica de color del Gráfico 1. La audiencia ya aprendió a leer esa forma, de modo que en la narrativa no gasta atención en decodificarla y puede concentrarse en el argumento.
* **Un solo color de énfasis a lo largo de los tres actos:** el rojo marca siempre el problema y el azul siempre la referencia o el escenario deseado. El significado del color no cambia entre paneles, lo que evita reaprendizajes.
* **Cierre con una cifra accionable:** el tercer acto abandona los porcentajes y vuelve a unidades que la audiencia gestiona (suscripciones y dólares), porque una narrativa dirigida a un equipo de decisión debe terminar en una magnitud sobre la que se pueda comprometer una meta.
* **Texto en cursiva bajo cada panel:** funciona como pie de acto y encadena la lectura, de modo que los tres paneles se perciban como un relato continuo y no como tres gráficos independientes.

<a id="sec-9"></a>
# 9. Conclusiones y recomendaciones basadas en los datos

## 9.1 Conclusiones

1. **La cancelación es un fenómeno concentrado, no transversal.** El 27,67% de cancelación global esconde un rango que va desde 11,99% en Premium hasta 44,14% en Básico. Cualquier acción de retención aplicada de forma uniforme desperdiciaría esfuerzo sobre los segmentos que no presentan el problema.

2. **El predictor relevante es el nivel de uso, no la preferencia de contenido.** Los usuarios que cancelan registran cerca de un 58% menos de reproducciones y un 60% menos de minutos por usuario, mientras que la distribución de géneros y de tipos de contenido es prácticamente idéntica entre ambos grupos. El catálogo no es el problema.

3. **No hay evidencia de un problema de experiencia de contenido.** El porcentaje promedio de contenido completado es de 71,52% en activos y 71,58% en cancelados, y la tasa de abandono temprano es igualmente baja en ambos grupos. Quien ve, termina lo que ve; el punto crítico es que ve poco.

4. **La experiencia técnica es una señal secundaria y consistente.** El buffering promedio es equivalente (8,26 frente a 8,24 segundos) y la distribución de dispositivos no diferencia a los grupos. Solo la calidad de video muestra una diferencia moderada, coherente con un menor uso general más que con una falla del servicio.

5. **El motivo declarado valida el diagnóstico cuantitativo.** "Poco uso" es la razón más frecuente de cancelación, seguida por "Precio". Ambas apuntan al mismo segmento: usuarios de bajo consumo en el plan más económico.

6. **Existe un marcador operativo anticipado.** La ausencia de renovación automática se concentra en el plan Básico y permite construir una lista de usuarios en riesgo antes de que la suscripción termine, sin necesidad de modelos predictivos adicionales.

## 9.2 Recomendaciones

| # | Recomendación | Evidencia que la sustenta | Indicador de seguimiento |
|---|---|---|---|
| 1 | Activar un programa de reactivación dirigido a usuarios del plan Básico con consumo bajo, antes del cierre del ciclo de facturación | Básico: 44,14% de cancelación; brecha de consumo de 58% | Tasa de cancelación mensual del plan Básico |
| 2 | Definir un umbral operativo de riesgo (por ejemplo, menos de 12 reproducciones al mes) y monitorearlo semanalmente | Los usuarios que cancelan promedian 11,8 reproducciones frente a 18,5 de los activos | Número de usuarios bajo el umbral y su evolución |
| 3 | Priorizar recomendaciones personalizadas por sobre la ampliación del catálogo | Las distribuciones de género y tipo de contenido no difieren entre grupos | Reproducciones por usuario en el plan Básico |
| 4 | Incentivar la activación de la renovación automática en el plan Básico, donde se concentra su ausencia | La cancelación es prácticamente total sin renovación automática | Porcentaje de usuarios con renovación automática por plan |
| 5 | Evaluar una migración asistida de Básico a Estándar para usuarios con uso creciente, en lugar de descuentos generalizados | "Precio" es el segundo motivo declarado, pero Estándar cancela menos de la mitad que Básico | Tasa de migración y cancelación posterior a los 3 meses |
| 6 | Mantener la experiencia técnica en monitoreo, sin asignarle prioridad de inversión | Buffering y dispositivos equivalentes entre ambos grupos | Buffering promedio por estado de suscripción |

## 9.3 Meta propuesta

Llevar la tasa de cancelación del plan Básico desde el 44,14% actual al nivel del plan Estándar (20,67%) permitiría retener del orden de 130 suscripciones, equivalentes a cerca de US$ 10.900 anuales de ingreso recurrente, sin requerir inversión en catálogo ni en infraestructura técnica. El seguimiento de esta meta se realiza directamente sobre el dashboard de la sección 7, filtrando por el plan Básico.